## Minimal setup - zeroshot deepforest
### Data:
- Data source: Settore innovazione digitale, dati, tecnologia e polo archivistico 
  (Regione Emilia-Romagna), Bologna RGB 2023
- License: CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/
- Changes: Multiple TIF tiles merged into a single mosaic covering the full 
  Bologna area, then used as input for zero-shot tree crown detection.

### How to run:
Kaggle notebook:
- Set up GPU (Settings → Accelerator → GPU T4x2)
- Turn on internet in the setting

### Results:
- Output can be visualised in QGIS using the original orthophoto and predictions:
  - `bologna_rgb_2023.tif` — merged RGB orthophoto
  - `tree_crowns_deepforest.gpkg` — predicted tree crown polygons

In [ ]:
!pip install deepforest

In [ ]:
from deepforest import main
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
import rasterio

In [ ]:

m = main.deepforest()
m.load_model(model_name="weecology/deepforest-tree", revision="main")
m.config["score_thresh"] = 0.1  

predictions = m.predict_tile(
    path="/kaggle/input/datasets/tercasaskova/bologna-tiff/bologna_rgb_2023.tif",
    patch_size=400,
    patch_overlap=0.10,
    iou_threshold=0.15,
    dataloader_strategy="window",  # memory safe — never loads full image
)

# Convert bbox columns to geometry
gdf = gpd.GeoDataFrame(
    predictions,
    geometry=[box(row.xmin, row.ymin, row.xmax, row.ymax) for _, row in predictions.iterrows()],
    crs="EPSG:4326"  # ← set to your actual TIF CRS
)

# Save directly to GPKG
gdf.to_file("tree_crowns_deepforest.gpkg", driver="GPKG")
print("saved to tree_crowns_deepforest.gpkg")

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

410030 predictions in overlapping windows, applying non-max suppression
331162 predictions kept after non-max suppression
      xmin     ymin     xmax     ymax label     score            image_path  \
0  51389.0  10637.0  51461.0  10703.0  Tree  0.956489  bologna_rgb_2023.tif   
1    583.0  12147.0    646.0  12211.0  Tree  0.939401  bologna_rgb_2023.tif   
2  35972.0  26637.0  36010.0  26674.0  Tree  0.938057  bologna_rgb_2023.tif   
3  23944.0  12712.0  24009.0  12774.0  Tree  0.935160  bologna_rgb_2023.tif   
4  66966.0  11199.0  67036.0  11267.0  Tree  0.935005  bologna_rgb_2023.tif   

                                            geometry  
0  POLYGON ((51461 10637, 51461 10703, 51389 1070...  
1  POLYGON ((646 12147, 646 12211, 583 12211, 583...  
2  POLYGON ((36010 26637, 36010 26674, 35972 2667...  
3  POLYGON ((24009 12712, 24009 12774, 23944 1277...  
4  POLYGON ((67036 11199, 67036 11267, 66966 1126...  
331,162 detections
